# Notebook 7 — Cleaned robustness analysis + fine-tuned/Instruct ensemble (CPU only)

1. **Robustness, cleaned.** Text-only exclusion rules fixed *before* re-analysis (no scores used): paraphrases with meta text, of originals ≤ 3 words, or more than 2× the original's length; typo variants identical to the original. Negation results are also split by form (*not* after an auxiliary vs. the "It is not the case that" prefix).
2. **Ensemble of the fine-tuned model and Instruct zero-shot**, two versions: a fixed 50/50 average (no fitting), and weights fitted on validation answers only. Evaluated on the real test sets (with paired tests vs. fine-tuned alone) and on the robustness sets.

**Before running:** upload `perturb.py` to `MyDrive/asag_revision/`. CPU runtime, ~10 min.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)

Mounted at /content/drive


In [2]:
import numpy as np, pandas as pd, nltk
from sklearn.linear_model import LinearRegression, LogisticRegression
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)
from asag_utils import *
from llm_utils import SCORE_TARGETS, LABEL_ORDER, score_from_logprobs, labels_from_logprobs
from perturb import valid_pairs, negation_form
RES = f'{ROOT}/results'
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
LPM = [f'lp_{s}' for s in SCORE_TARGETS]; LPS = [f'lp_{l}' for l in LABEL_ORDER]
VARIANTS = ['negated', 'keyword_salad', 'swapped', 'typos', 'paraphrase']

def cluster_ci(values, groups, n_boot=2000, seed=SEED):
    values, groups = np.asarray(values, float), np.asarray(groups)
    sets_ = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    rng = np.random.default_rng(seed)
    b = [values[np.concatenate([sets_[i] for i in rng.integers(0, len(sets_), len(sets_))])].mean() for _ in range(n_boot)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

def paired_bootstrap(y, a, b, metric_fn, groups=None, n_boot=2000, seed=SEED):
    y, a, b = np.asarray(y), np.asarray(a), np.asarray(b)
    rng = np.random.default_rng(seed)
    sets_ = None if groups is None else [np.flatnonzero(np.asarray(groups) == g) for g in pd.unique(groups)]
    ma, mb = metric_fn(y, a), metric_fn(y, b); D = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y)) if sets_ is None else np.concatenate([sets_[i] for i in rng.integers(0, len(sets_), len(sets_))])
        A, B = metric_fn(y[idx], a[idx]), metric_fn(y[idx], b[idx]); D.append({k: A[k] - B[k] for k in A})
    D = pd.DataFrame(D)
    return {k: (ma[k] - mb[k], np.percentile(D[k].dropna(), 2.5), np.percentile(D[k].dropna(), 97.5),
                min(1.0, max(2 * min((D[k] <= 0).mean(), (D[k] >= 0).mean()), 1 / n_boot))) for k in ma}

## 1. Mohler ensemble: fitted on each fold's validation answers

In [3]:
ft = pd.concat([pd.read_csv(f'{RES}/mohler_llama3_ft_f{k}.csv') for k in range(N_FOLDS)])
zs = pd.read_csv(f'{RES}/mohler_llama3_instruct_zs.csv').drop_duplicates('uid').set_index('uid').expected
preds = pd.read_csv(f'{RES}/mohler_preds.csv')
ens_w, fitted = [], {}
for k in range(N_FOLDS):
    v = ft[(ft.fold == k) & (ft.split == 'val')].merge(mo[['uid', 'y']], on='uid')
    v['zs'] = v.uid.map(zs)
    lr = LinearRegression(positive=True).fit(v[['expected', 'zs']], v.y); fitted[k] = lr
    w = lr.coef_ / lr.coef_.sum() if lr.coef_.sum() > 0 else lr.coef_
    ens_w.append({'fold': k, 'w_finetuned': w[0], 'w_instruct': w[1], 'intercept': lr.intercept_})
ens_w = pd.DataFrame(ens_w); ens_w.to_csv(f'{RES}/nb7_mohler_ensemble_weights.csv', index=False)
display(ens_w.round(3))

test = ft[ft.split == 'test'].set_index('uid')
preds['ens_50_50'] = 0.5 * preds.llama3_ft + 0.5 * preds.llama3_instruct_zs
preds['ens_fitted'] = [np.clip(fitted[f].predict(pd.DataFrame({'expected': [e], 'zs': [z]}))[0], 0, 10)
                       for f, e, z in zip(preds.fold, preds.llama3_ft, preds.llama3_instruct_zs)]
preds.to_csv(f'{RES}/mohler_preds.csv', index=False)
rows = [{'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)}
        for s in ['llama3_ft', 'llama3_instruct_zs', 'ens_50_50', 'ens_fitted']]
ens_m = pd.DataFrame(rows); ens_m.to_csv(f'{RES}/mohler_nb7_ensemble_summary.csv', index=False)
display(ens_m[['system', 'pearson', 'pearson_ci', 'mae', 'rmse', 'qwk', 'pearson_fold_std']].round(3))
tests = []
for s in ['ens_50_50', 'ens_fitted']:
    r = paired_bootstrap(preds.y, preds[s], preds.llama3_ft, reg_metrics, groups=preds.group)
    tests.append({'comparison': f'{s} vs llama3_ft', **{f'd_{m}': r[m][0] for m in ['pearson', 'mae', 'qwk']},
                  'd_pearson_ci': f'[{r["pearson"][1]:.3f}, {r["pearson"][2]:.3f}]', 'p_pearson': r['pearson'][3],
                  'd_mae_ci': f'[{r["mae"][1]:.3f}, {r["mae"][2]:.3f}]', 'p_mae': r['mae'][3]})
display(pd.DataFrame(tests).round(4))

,fold,w_finetuned,w_instruct,intercept
0,0,0.843,0.157,-2.216
1,1,1.000,0.000,0.677
2,2,0.973,0.027,-2.675
3,3,1.000,0.000,-1.166
4,4,0.865,0.135,-1.331


,system,pearson,pearson_ci,mae,rmse,qwk,pearson_fold_std
0,llama3_ft,0.765,"[0.718, 0.809]",0.969,1.433,0.747,0.039
1,llama3_instruct_zs,0.673,"[0.623, 0.725]",1.625,2.116,0.600,0.073
2,ens_50_50,0.762,"[0.716, 0.803]",1.184,1.554,0.714,0.055
3,ens_fitted,0.789,"[0.751, 0.822]",0.977,1.407,0.777,0.039


,comparison,d_pearson,d_mae,d_qwk,d_pearson_ci,p_pearson,d_mae_ci,p_mae
0,ens_50_50 vs llama3_ft,-0.0033,0.2144,-0.0335,"[-0.021, 0.013]",0.703,"[0.157, 0.270]",0.0005
1,ens_fitted vs llama3_ft,0.0232,0.0076,0.0304,"[0.007, 0.040]",0.006,"[-0.034, 0.051]",0.7570


## 2. SciEntsBank ensemble: logistic regression on both models' label log-probabilities (fitted on validation), and a 50/50 probability average

In [4]:
ftS = pd.read_csv(f'{RES}/scientsbank_llama3_ft.csv').drop_duplicates('uid').merge(se[['uid', 'split', 'group', 'label', 'label3', 'label2']], on='uid')
zsS = pd.read_csv(f'{RES}/scientsbank_llama3_instruct_zs.csv').drop_duplicates('uid').set_index('uid')
calS = zsS.join(se.set_index('uid')[['split', 'label']]); calS = calS[calS.split.isin(['train', 'val'])]
qc, _, _ = labels_from_logprobs(calS[LPS].values)
zs_cal = LogisticRegression(max_iter=2000).fit(np.log(np.clip(qc, 1e-12, 1)), calS.label)   # Notebook 2 calibration

def probs_aligned(clf, X):
    p = np.zeros((len(X), len(LABEL_ORDER)))
    for j, c in enumerate(clf.classes_):
        p[:, LABEL_ORDER.index(c)] = clf.predict_proba(X)[:, j]
    return p

def ens_features(lp_ft, lp_zs):
    q1, _, _ = labels_from_logprobs(lp_ft); q2, _, _ = labels_from_logprobs(lp_zs)
    return q1, probs_aligned(zs_cal, np.log(np.clip(q2, 1e-12, 1)))

q_ft, q_zs = ens_features(ftS[LPS].values, zsS.loc[ftS.uid, LPS].values)
X = np.log(np.clip(np.hstack([q_ft, q_zs]), 1e-12, 1))
val = (ftS.split == 'val').values
ens_clf = LogisticRegression(max_iter=4000).fit(X[val], ftS.label.values[val])
_, _, p_fit = labels_from_logprobs(np.log(np.clip(probs_aligned(ens_clf, X), 1e-12, 1)))
_, _, p_avg = labels_from_logprobs(np.log(np.clip(0.5 * q_ft + 0.5 * q_zs, 1e-12, 1)))
sp = pd.read_csv(f'{RES}/scientsbank_preds.csv')
new = pd.DataFrame({'uid': ftS.uid})
for sch in ['5way', '3way', '2way']:
    new[f'ens_fitted_{sch}'] = p_fit[sch]; new[f'ens_50_50_{sch}'] = p_avg[sch]
sp = sp.drop(columns=[c for c in new.columns if c != 'uid' and c in sp]).merge(new, on='uid', how='left')
sp.to_csv(f'{RES}/scientsbank_preds.csv', index=False)

label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
rows, tests = [], []
for sch, col in label_col.items():
    for t in ['test_ua', 'test_uq', 'test_ud']:
        cols = [f'{s}_{sch}' for s in ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted']]
        d = sp[sp.split == t].dropna(subset=cols); g = None if t == 'test_ua' else d.group
        for s in ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted']:
            m = cls_metrics(d[col], d[f'{s}_{sch}'])
            rows.append({'scheme': sch, 'test': t, 'system': s, **m})
        for s in ['ens_50_50', 'ens_fitted']:
            r = paired_bootstrap(d[col], d[f'{s}_{sch}'], d[f'llama3_ft_{sch}'], cls_metrics, groups=g)
            tests.append({'scheme': sch, 'test': t, 'comparison': f'{s} vs llama3_ft',
                          'd_macro_f1': r['macro_f1'][0], 'ci': f'[{r["macro_f1"][1]:.3f}, {r["macro_f1"][2]:.3f}]', 'p': r['macro_f1'][3],
                          'd_weighted_f1': r['weighted_f1'][0], 'p_weighted': r['weighted_f1'][3]})
ens_s = pd.DataFrame(rows); ens_s.to_csv(f'{RES}/scientsbank_nb7_ensemble_summary.csv', index=False)
pd.DataFrame(tests).to_csv(f'{RES}/scientsbank_nb7_ensemble_tests.csv', index=False)
for metric in ['macro_f1', 'weighted_f1']:
    print(metric); display(ens_s.pivot_table(index=['scheme', 'system'], columns='test', values=metric).round(3))
display(pd.DataFrame(tests).round(4))

macro_f1


test                           test_ua  test_ud  test_uq
scheme system                                           
2way   ens_50_50                 0.831    0.765    0.832
       ens_fitted                0.843    0.769    0.827
       llama3_ft                 0.830    0.769    0.820
       llama3_instruct_zs_cal    0.747    0.712    0.806
3way   ens_50_50                 0.775    0.733    0.741
       ens_fitted                0.787    0.738    0.743
       llama3_ft                 0.775    0.733    0.746
       llama3_instruct_zs_cal    0.622    0.567    0.652
5way   ens_50_50                 0.585    0.717    0.524
       ens_fitted                0.675    0.640    0.519
       llama3_ft                 0.581    0.720    0.522
       llama3_instruct_zs_cal    0.523    0.572    0.418

weighted_f1


test                           test_ua  test_ud  test_uq
scheme system                                           
2way   ens_50_50                 0.836    0.771    0.838
       ens_fitted                0.846    0.773    0.833
       llama3_ft                 0.834    0.775    0.828
       llama3_instruct_zs_cal    0.752    0.717    0.813
3way   ens_50_50                 0.792    0.733    0.789
       ens_fitted                0.798    0.736    0.780
       llama3_ft                 0.789    0.737    0.781
       llama3_instruct_zs_cal    0.680    0.636    0.741
5way   ens_50_50                 0.748    0.661    0.679
       ens_fitted                0.738    0.645    0.675
       llama3_ft                 0.738    0.667    0.676
       llama3_instruct_zs_cal    0.587    0.540    0.583

,scheme,test,comparison,d_macro_f1,ci,p,d_weighted_f1,p_weighted
0,5way,test_ua,ens_50_50 vs llama3_ft,0.0041,"[-0.011, 0.019]",0.5570,0.0097,0.237
1,5way,test_ua,ens_fitted vs llama3_ft,0.0947,"[-0.023, 0.202]",0.5130,0.0000,0.985
2,5way,test_uq,ens_50_50 vs llama3_ft,0.0025,"[-0.016, 0.023]",0.7560,0.0035,0.720
3,5way,test_uq,ens_fitted vs llama3_ft,-0.0031,"[-0.139, 0.019]",0.5650,-0.0008,0.980
4,5way,test_ud,ens_50_50 vs llama3_ft,-0.0029,"[-0.017, 0.011]",0.6550,-0.0068,0.326
5,5way,test_ud,ens_fitted vs llama3_ft,-0.0799,"[-0.124, -0.044]",0.0005,-0.0221,0.051
6,3way,test_ua,ens_50_50 vs llama3_ft,-0.0002,"[-0.017, 0.017]",0.9720,0.0036,0.664
7,3way,test_ua,ens_fitted vs llama3_ft,0.0116,"[-0.008, 0.033]",0.2610,0.0091,0.353
8,3way,test_uq,ens_50_50 vs llama3_ft,-0.0048,"[-0.027, 0.017]",0.6710,0.0085,0.234
9,3way,test_uq,ens_fitted vs llama3_ft,-0.0028,"[-0.024, 0.034]",0.9030,-0.0006,0.968


## 3. Robustness, cleaned, with the ensembles added

In [5]:
rsM = pd.read_csv(f'{RES}/robustness_set.csv'); rsS = pd.read_csv(f'{RES}/robustness_set_scientsbank.csv')
for name, rs in [('mohler', rsM), ('scientsbank', rsS)]:
    ok = valid_pairs(rs)
    print(name, 'excluded:', rs[~ok].variant.value_counts().to_dict())
    rs['keep'] = ok.values
    orig = rs[rs.variant == 'original'].set_index('orig_uid').student_answer
    rs['neg_form'] = rs.orig_uid.map(orig).astype(str).map(negation_form)

# --- Mohler scores
S = rsM[['rid', 'orig_uid', 'fold', 'group', 'variant', 'keep', 'neg_form']].copy()
for name, f in [('llama3_ft', 'robustness_scores_ft.csv'), ('llama3_instruct_zs', 'robustness_scores_instruct_zs.csv')]:
    z = pd.read_csv(f'{RES}/{f}'); e, _, _ = score_from_logprobs(z[LPM].values)
    S = S.merge(pd.DataFrame({'rid': z.rid, name: e}), on='rid')
S = S.merge(pd.read_csv(f'{RES}/robustness_scores_lexical.csv')[['rid', 'score']].rename(columns={'score': 'lexical_composite'}), on='rid')
S['ens_50_50'] = 0.5 * S.llama3_ft + 0.5 * S.llama3_instruct_zs
S['ens_fitted'] = [np.clip(fitted[f].predict(pd.DataFrame({'expected': [a], 'zs': [b]}))[0], 0, 10)
                   for f, a, b in zip(S.fold, S.llama3_ft, S.llama3_instruct_zs)]
SYS_M = ['llama3_ft', 'llama3_instruct_zs', 'ens_50_50', 'ens_fitted', 'lexical_composite']
orig = S[S.variant == 'original'].set_index('orig_uid')
rows = []
for s in SYS_M:
    for v in VARIANTS + ['negated:auxiliary', 'negated:prefix']:
        base_v, form = (v.split(':') + [None])[:2]
        dv = S[(S.variant == base_v) & S.keep]
        if form: dv = dv[dv.neg_form == form]
        dv = dv.set_index('orig_uid'); delta = dv[s] - orig.loc[dv.index, s]
        lo, hi = cluster_ci(delta.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'n': len(dv), 'mean_delta': delta.mean(), 'delta_ci': f'[{lo:.2f}, {hi:.2f}]',
                     'share_drop_ge3': (delta <= -3).mean(), 'share_abs_delta_le1': (delta.abs() <= 1).mean()})
robM = pd.DataFrame(rows); robM.to_csv(f'{RES}/robustness_summary_clean.csv', index=False)
display(robM.pivot_table(index='variant', columns='system', values='mean_delta').loc[VARIANTS + ['negated:auxiliary', 'negated:prefix'], SYS_M].round(2))
display(robM.round(3))

/content/drive/MyDrive/asag_revision/perturb.py:52: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bad_para = (rs.variant == "paraphrase") & (rs.student_answer.astype(str).str.contains(META) | (n_o <= 3) | (n_v > 2 * n_o))
/content/drive/MyDrive/asag_revision/perturb.py:52: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bad_para = (rs.variant == "paraphrase") & (rs.student_answer.astype(str).str.contains(META) | (n_o <= 3) | (n_v > 2 * n_o))


mohler excluded: {'paraphrase': 49, 'typos': 9}
scientsbank excluded: {'paraphrase': 29, 'typos': 8}


system,llama3_ft,llama3_instruct_zs,ens_50_50,ens_fitted,lexical_composite
variant,,,,,
negated,-2.18,-4.82,-3.50,-2.70,-0.10
keyword_salad,0.21,-2.42,-1.10,0.02,1.22
swapped,-4.29,-7.60,-5.95,-5.15,-2.35
typos,-0.13,-0.77,-0.45,-0.20,-0.85
paraphrase,-0.20,0.09,-0.05,-0.20,-0.12
negated:auxiliary,-1.25,-2.64,-1.95,-1.58,-0.10
negated:prefix,-3.09,-6.95,-5.02,-3.81,-0.09


,system,variant,n,mean_delta,delta_ci,share_drop_ge3,share_abs_delta_le1
0,llama3_ft,negated,200,-2.175,"[-2.60, -1.78]",0.340,0.370
1,llama3_ft,keyword_salad,200,0.206,"[0.02, 0.39]",0.000,0.810
2,llama3_ft,swapped,200,-4.289,"[-4.56, -4.01]",0.825,0.025
3,llama3_ft,typos,191,-0.135,"[-0.27, -0.05]",0.010,0.984
4,llama3_ft,paraphrase,151,-0.198,"[-0.34, -0.08]",0.020,0.901
5,llama3_ft,negated:auxiliary,99,-1.247,"[-1.64, -0.87]",0.162,0.606
6,llama3_ft,negated:prefix,101,-3.086,"[-3.58, -2.52]",0.515,0.139
7,llama3_instruct_zs,negated,200,-4.818,"[-5.60, -4.07]",0.610,0.270
8,llama3_instruct_zs,keyword_salad,200,-2.416,"[-3.01, -1.83]",0.365,0.300
9,llama3_instruct_zs,swapped,200,-7.602,"[-7.98, -7.10]",0.930,0.035


In [6]:
# --- SciEntsBank labels
P = rsS[['rid', 'orig_uid', 'group', 'variant', 'keep', 'neg_form']].copy()
zf = pd.read_csv(f'{RES}/robustness_scores_scientsbank_ft.csv').set_index('rid')
zz = pd.read_csv(f'{RES}/robustness_scores_scientsbank_instruct_zs.csv').set_index('rid')
q1, q2 = ens_features(zf.loc[P.rid, LPS].values, zz.loc[P.rid, LPS].values)
for name, q in [('llama3_ft', q1), ('llama3_instruct_zs_cal', q2), ('ens_50_50', 0.5 * q1 + 0.5 * q2),
                ('ens_fitted', probs_aligned(ens_clf, np.log(np.clip(np.hstack([q1, q2]), 1e-12, 1))))]:
    P[name] = np.array(LABEL_ORDER)[q.argmax(1)]
P = P.merge(pd.read_csv(f'{RES}/robustness_scores_scientsbank_lexical.csv')[['rid', 'label_pred']].rename(columns={'label_pred': 'lexical_4'}), on='rid')
SYS_S = ['llama3_ft', 'llama3_instruct_zs_cal', 'ens_50_50', 'ens_fitted', 'lexical_4']
rows = []
for s in SYS_S:
    for v in ['original'] + VARIANTS + ['negated:auxiliary', 'negated:prefix']:
        base_v, form = (v.split(':') + [None])[:2]
        dv = P[(P.variant == base_v) & P.keep]
        if form: dv = dv[dv.neg_form == form]
        still = (dv[s] == 'correct').astype(float)
        lo, hi = cluster_ci(still.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'n': len(dv), 'share_still_correct': still.mean(), 'ci': f'[{lo:.2f}, {hi:.2f}]',
                     'share_contradictory': (dv[s] == 'contradictory').mean(),
                     'share_irrelevant_or_nondomain': dv[s].isin(['irrelevant', 'non_domain']).mean()})
robS = pd.DataFrame(rows); robS.to_csv(f'{RES}/robustness_summary_scientsbank_clean.csv', index=False)
display(robS.pivot_table(index='variant', columns='system', values='share_still_correct')
        .loc[['original'] + VARIANTS + ['negated:auxiliary', 'negated:prefix'], SYS_S].round(3))
display(robS.round(3))

system,llama3_ft,llama3_instruct_zs_cal,ens_50_50,ens_fitted,lexical_4
variant,,,,,
original,0.720,0.760,0.725,0.755,0.860
negated,0.310,0.245,0.290,0.465,0.855
keyword_salad,0.705,0.150,0.630,0.785,1.000
swapped,0.005,0.005,0.005,0.005,0.055
typos,0.698,0.661,0.714,0.766,0.688
paraphrase,0.696,0.813,0.731,0.789,0.825
negated:auxiliary,0.381,0.324,0.367,0.453,0.885
negated:prefix,0.148,0.066,0.115,0.492,0.787


,system,variant,n,share_still_correct,ci,share_contradictory,share_irrelevant_or_nondomain
0,llama3_ft,original,200,0.720,"[0.63, 0.80]",0.020,0.030
1,llama3_ft,negated,200,0.310,"[0.23, 0.40]",0.490,0.050
2,llama3_ft,keyword_salad,200,0.705,"[0.59, 0.81]",0.075,0.010
3,llama3_ft,swapped,200,0.005,"[0.00, 0.02]",0.035,0.935
4,llama3_ft,typos,192,0.698,"[0.61, 0.78]",0.026,0.042
5,llama3_ft,paraphrase,171,0.696,"[0.60, 0.79]",0.029,0.035
6,llama3_ft,negated:auxiliary,139,0.381,"[0.29, 0.48]",0.424,0.022
7,llama3_ft,negated:prefix,61,0.148,"[0.06, 0.26]",0.639,0.115
8,llama3_instruct_zs_cal,original,200,0.760,"[0.69, 0.83]",0.040,0.075
9,llama3_instruct_zs_cal,negated,200,0.245,"[0.17, 0.33]",0.590,0.070


Mohler table: mean change in score (lower is better for negated/keyword_salad/swapped; ≈0 is better for typos/paraphrase). SciEntsBank table: share still labelled `correct` (lower is better for the meaning-destroying variants; close to the `original` row is better for typos/paraphrase).